# AgroPredix - Entrenamiento del modelo
Modelo hibrido: clasificador ML + sistema difuso aprendido de datos (Wang-Mendel) + ajuste bayesiano con evidencia local.
Entrada: `sisagri_modelo.csv`, `sisagri_campana.csv`, `sisagri_mensual_limpio.csv`

## 0. Configuracion

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, brier_score_loss
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)

SEED, N_FOLDS = 42, 5
MIN_PROV_REG = 5      # registros minimos de un cultivo en la provincia para recomendarlo
BASE = "https://raw.githubusercontent.com/AldoVasquez07/st2-rec-doc-2026/refs/heads/main/Dataset/"

## 1. Carga de datos

In [ ]:
def leer(nombre):
    if os.path.exists(nombre):                       # archivo subido a la sesion
        return pd.read_csv(nombre, encoding="utf-8-sig")
    try:                                             # repositorio
        return pd.read_csv(BASE + nombre, encoding="utf-8-sig")
    except Exception:
        from google.colab import files
        print("Sube el archivo:", nombre)
        up = files.upload()
        return pd.read_csv(next(iter(up)), encoding="utf-8-sig")

modelo   = leer("sisagri_modelo.csv")
camp     = leer("sisagri_campana.csv")
mensual  = leer("sisagri_mensual_limpio.csv")
print("modelo", modelo.shape, "| campana", camp.shape, "| mensual", mensual.shape)
modelo.groupby("campana").exito.agg(["size", "mean"])

## 2. Variables permitidas y fuga de informacion

In [ ]:
VARIABLES = pd.DataFrame([
    ("provincia, producto, familia",                       "decision",  "donde y que se siembra"),
    ("mes_pico_siembra",                                   "decision",  "cuando se siembra"),
    ("siembra_ha",                                         "decision",  "area planificada"),
    ("prev_* (campana previa del distrito)",               "historia",  "rotacion a nivel distrito"),
    ("cosecha_ha, produccion_t, precio_kg, n_meses_activo", "resultado", "se conocen al cosechar: fuga"),
    ("rendimiento_t_ha, rend_relativo, etiqueta_*, exito",  "resultado", "definen la etiqueta: fuga"),
    ("ingreso_ha_soles, ratio_cosecha_siembra, tasa_perdida_camp", "resultado", "se conocen al cosechar: fuga"),
], columns=["variables", "tipo", "motivo"])
VARIABLES

## 3. Ingenieria de variables

In [ ]:
m = modelo.copy()

# 3.1 escala y saturacion del cultivo en el distrito (tabla de campana)
tr = camp[camp.tipo_cultivo == "transitorio"]
tot = tr.groupby(["cod_ubigeo", "campana"]).siembra_ha.sum().rename("sup_total_dist").reset_index()
m = m.merge(tot, on=["cod_ubigeo", "campana"], how="left")
m["share_cultivo_dist"] = (m.siembra_ha / m.sup_total_dist.replace(0, np.nan)).fillna(0)
m["log_siembra"] = np.log1p(m.siembra_ha)
m["log_prev_cultivo"] = np.log1p(m.prev_siembra_cultivo)

# 3.2 calendario tipico de siembra por cultivo (tabla mensual)
s = mensual[mensual.siembra_ha > 0].groupby(["producto", "mes"]).siembra_ha.sum().unstack(fill_value=0)
MES_TIPICO = s.idxmax(axis=1)

def dist_circular(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 12 - d)

m["mes_tipico"] = m.producto.map(MES_TIPICO)
m["dist_mes_tipico"] = dist_circular(m.mes_pico_siembra, m.mes_tipico)
m["sin_siembra"] = m.mes_pico_siembra.isna().astype(int)
m["mes_sin"] = np.sin(2 * np.pi * m.mes_pico_siembra / 12)
m["mes_cos"] = np.cos(2 * np.pi * m.mes_pico_siembra / 12)

CAT     = ["provincia", "producto", "familia"]
NUM_MES = ["mes_sin", "mes_cos", "sin_siembra", "dist_mes_tipico"]
NUM_ESC = ["log_siembra", "share_cultivo_dist"]
NUM_ROT = ["prev_share_familia", "prev_share_leguminosa", "log_prev_cultivo"]
NUM = NUM_MES + NUM_ESC + NUM_ROT
FEATS = CAT + NUM

y = m.exito.astype(int).values
g = m.cod_ubigeo.values
yrel = m.rend_relativo.clip(upper=3).values
SPLITS = list(GroupKFold(n_splits=N_FOLDS).split(m, y, g))     # validacion por distrito
print("Filas:", len(m), "| Distritos:", len(set(g)), "| Exito:", y.mean().round(3))
m[FEATS].isna().sum()[lambda s: s > 0]

## 4. Validacion y seleccion de modelo

### 4.1 Utilidades

In [ ]:
def make_pipe(cat, num, model):
    pre = ColumnTransformer([
        ("c", OneHotEncoder(handle_unknown="ignore"), cat),
        ("n", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num)], sparse_threshold=0)
    return make_pipeline(pre, model)

def metricas(y_true, p):
    return dict(auc=roc_auc_score(y_true, p), acc=accuracy_score(y_true, p >= .5),
                f1=f1_score(y_true, p >= .5), brier=brier_score_loss(y_true, p))

def cv_oof(cat, num, make_model):
    oof = np.zeros(len(m))
    for a, b in SPLITS:
        p = make_pipe(cat, num, make_model()).fit(m.iloc[a][cat + num], y[a])
        oof[b] = p.predict_proba(m.iloc[b][cat + num])[:, 1]
    return oof

RF = lambda **kw: RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED, **kw)

### 4.2 Comparacion de modelos (validacion cruzada agrupada por distrito)

In [ ]:
modelos = {
    "Logistica":         lambda: LogisticRegression(max_iter=3000, C=0.5),
    "Random Forest":     lambda: RF(min_samples_leaf=5),
    "Gradient Boosting": lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=150, random_state=SEED),
}
filas = [dict(modelo="Constante (prior)", **metricas(y, np.full(len(y), y.mean())))]
oofs = {}
for nombre, fm in modelos.items():
    oofs[nombre] = cv_oof(CAT, NUM, fm)
    filas.append(dict(modelo=nombre, **metricas(y, oofs[nombre])))
comp = pd.DataFrame(filas).set_index("modelo").round(3)
display(comp)

comp.drop("Constante (prior)").auc.plot.barh(figsize=(6, 2.5), color="#4c9a2a", xlim=(.5, 1))
plt.title("AUC por modelo (CV agrupada por distrito)"); plt.show()

### 4.3 Ablacion por grupos de variables (Random Forest)

In [ ]:
# AUC < 0.5 en "solo cultivo" es esperado: la etiqueta es relativa a la mediana de cada cultivo
pasos = [("solo cultivo", ["producto", "familia"], []),
         ("+ provincia", CAT, []),
         ("+ calendario de siembra", CAT, NUM_MES),
         ("+ escala del cultivo", CAT, NUM_MES + NUM_ESC),
         ("+ rotacion (campana previa)", CAT, NUM)]
ab = []
for nombre, c_, n_ in pasos:
    ab.append(dict(variables=nombre, **metricas(y, cv_oof(c_, n_, lambda: RF(min_samples_leaf=5)))))
pd.DataFrame(ab).set_index("variables").round(3)

### 4.4 Verificacion temporal (entrenar 2015-16, probar 2016-17)

In [ ]:
ent = (m.campana == "2015-2016").values
num_t = NUM_MES + NUM_ESC            # sin rotacion: no existe campana previa en 2015-16
p = make_pipe(CAT, num_t, RF(min_samples_leaf=5)).fit(m[ent][CAT + num_t], y[ent])
pt = p.predict_proba(m[~ent][CAT + num_t])[:, 1]
pd.Series(metricas(y[~ent], pt)).round(3).to_frame("temporal 2016-17")

## 5. Modelo ML final

### 5.1 Ajuste de hiperparametros

In [ ]:
pipe = make_pipe(CAT, NUM, RandomForestClassifier(n_estimators=300, n_jobs=1, random_state=SEED))
grid = {"randomforestclassifier__max_depth": [6, None],
        "randomforestclassifier__min_samples_leaf": [3, 5, 10],
        "randomforestclassifier__max_features": ["sqrt", 0.5]}
gs = GridSearchCV(pipe, grid, cv=SPLITS, scoring="roc_auc", n_jobs=-1).fit(m[FEATS], y)
BEST = {k.split("__")[1]: v for k, v in gs.best_params_.items()}
print(BEST, "| AUC CV:", round(gs.best_score_, 3))

### 5.2 Calibracion de probabilidades

In [ ]:
def oof_ml(calibrar):
    oof = np.zeros(len(m))
    for a, b in SPLITS:
        base = make_pipe(CAT, NUM, RF(**BEST))
        if calibrar:
            inner = list(GroupKFold(3).split(m.iloc[a], y[a], g[a]))
            base = CalibratedClassifierCV(base, method="sigmoid", cv=inner)
        oof[b] = base.fit(m.iloc[a][FEATS], y[a]).predict_proba(m.iloc[b][FEATS])[:, 1]
    return oof

oof_raw, oof_cal = oof_ml(False), oof_ml(True)
print(pd.DataFrame({"sin calibrar": metricas(y, oof_raw), "calibrado": metricas(y, oof_cal)}).round(3))

# se usa la version con menor Brier
CALIBRAR = brier_score_loss(y, oof_cal) < brier_score_loss(y, oof_raw)
oof_ml_sel = oof_cal if CALIBRAR else oof_raw
print("Calibracion aplicada:", CALIBRAR)

plt.figure(figsize=(4.5, 4.5))
for nombre, o in [("sin calibrar", oof_raw), ("calibrado", oof_cal)]:
    fr, mp = calibration_curve(y, o, n_bins=8)
    plt.plot(mp, fr, marker="o", label=nombre)
plt.plot([0, 1], [0, 1], "k--"); plt.xlabel("probabilidad predicha"); plt.ylabel("frecuencia observada")
plt.title("Curva de calibracion (OOF)"); plt.legend(); plt.show()

### 5.3 Importancia de variables (permutacion en distritos no vistos)

In [ ]:
imp = []
for a, b in SPLITS:
    p = make_pipe(CAT, NUM, RF(**BEST)).fit(m.iloc[a][FEATS], y[a])
    r = permutation_importance(p, m.iloc[b][FEATS], y[b], scoring="roc_auc", n_repeats=5, random_state=SEED)
    imp.append(r.importances_mean)
imp = pd.Series(np.mean(imp, axis=0), index=FEATS).sort_values()
imp.plot.barh(figsize=(7, 4.5), color="#2a6f9a")
plt.title("Caida de AUC al permutar cada variable"); plt.show()

## 6. Sistema difuso aprendido de datos (Wang-Mendel)

In [ ]:
FUZ_VARS = ["zona", "zona_cultivo", "ventana", "escala"]
TERMINOS = {"zona": ["baja", "media", "alta"], "zona_cultivo": ["baja", "media", "alta"],
            "ventana": ["en_epoca", "cercana", "fuera"], "escala": ["pequena", "mediana", "grande"]}
SAL_TERM = ["bajo", "medio", "alto"]

def fuzz3(x, c):
    # particion triangular de 3 terminos con centros c1 < c2 < c3
    c1, c2, c3 = c
    bajo = np.clip((c2 - x) / (c2 - c1), 0, 1)
    alto = np.clip((x - c2) / (c3 - c2), 0, 1)
    return np.stack([bajo, 1 - bajo - alto, alto], axis=1)

class Zona:
    # aptitud historica de la provincia y de la provincia-cultivo (suavizada hacia el prior)
    def __init__(self, d, k=5):
        self.gm = d.exito.mean()
        a = d.groupby("provincia").exito.agg(["sum", "count"])
        self.prov = ((a["sum"] + k * self.gm) / (a["count"] + k)).to_dict()
        b = d.groupby(["provincia", "producto"]).exito.agg(["sum", "count"])
        self.pc = {(p, c): (r["sum"] + k * self.prov[p]) / (r["count"] + k) for (p, c), r in b.iterrows()}
    def transform(self, d):
        z = d.provincia.map(self.prov).fillna(self.gm).values
        zc = np.array([self.pc.get((a, b), np.nan) for a, b in zip(d.provincia, d.producto)])
        zc = np.where(np.isnan(zc), z, zc)
        return np.column_stack([z, zc, d.dist_mes_tipico.fillna(3).values, d.log_siembra.values])

class DifusoWM:
    def fit(self, X, yv):
        self.cs = [np.array([0., 3., 6.]) if i == 2 else np.quantile(X[:, i], [.1, .5, .9]) for i in range(X.shape[1])]
        self.cy = np.quantile(yv, [.1, .5, .9])
        Mx = [fuzz3(X[:, i], self.cs[i]) for i in range(X.shape[1])]
        My = fuzz3(yv, self.cy)
        mejor = {}
        for n in range(len(yv)):
            ant = tuple(int(M[n].argmax()) for M in Mx)
            con = int(My[n].argmax())
            grado = np.prod([M[n].max() for M in Mx]) * My[n].max()
            if ant not in mejor or grado > mejor[ant][1]:
                mejor[ant] = (con, grado)
        self.reglas = {a: v[0] for a, v in mejor.items()}
        return self
    def fuerzas(self, X):
        Mx = [fuzz3(X[:, i], self.cs[i]) for i in range(X.shape[1])]
        return {a: np.prod([Mx[i][:, a[i]] for i in range(len(a))], axis=0) for a in self.reglas}
    def predict(self, X):
        F = self.fuerzas(X)
        num = sum(F[a] * self.cy[c] for a, c in self.reglas.items())
        den = sum(F.values())
        return np.where(den > 1e-9, num / np.maximum(den, 1e-9), self.cy[1])
    def texto(self, a):
        ant = " Y ".join(f"{n} es {TERMINOS[n][i]}" for n, i in zip(FUZ_VARS, a))
        return f"SI {ant} ENTONCES rendimiento es {SAL_TERM[self.reglas[a]]}"
    def top_reglas(self, X, k=3):
        F = self.fuerzas(X)
        orden = sorted(F, key=lambda a: -F[a][0])[:k]
        return [(self.texto(a), float(F[a][0])) for a in orden if F[a][0] > 0]

# validacion por distrito
oof_fz = np.zeros(len(m))
for a, b in SPLITS:
    zn = Zona(m.iloc[a]); Xa, Xb = zn.transform(m.iloc[a]), zn.transform(m.iloc[b])
    wm = DifusoWM().fit(Xa, yrel[a])
    cal = LogisticRegression().fit(wm.predict(Xa).reshape(-1, 1), y[a])
    oof_fz[b] = cal.predict_proba(wm.predict(Xb).reshape(-1, 1))[:, 1]
print("Reglas aprendidas (ultimo pliegue):", len(wm.reglas))
pd.Series(metricas(y, oof_fz)).round(3).to_frame("difuso (OOF)")

### 6.1 Combinacion con el modelo ML

In [ ]:
ws = [0, .25, .5, .75, 1]
tab = pd.DataFrame({w: metricas(y, w * oof_ml_sel + (1 - w) * oof_fz) for w in ws}).T.round(3)
tab.index.name = "peso ML"
W_ML = tab.brier.idxmin()
oof_mix = W_ML * oof_ml_sel + (1 - W_ML) * oof_fz
print("Peso ML elegido (menor Brier):", W_ML)
tab

## 7. Ajuste bayesiano con evidencia local

### 7.1 Persistencia del resultado entre campanas

In [ ]:
h = m.pivot_table(index=["cod_ubigeo", "producto"], columns="campana_ini", values="exito")
h = h.dropna()
h.columns = ["previo", "actual"]
print("Pares distrito-cultivo con ambas campanas:", len(h))
print(h.groupby("previo").actual.agg(["mean", "size"]).round(3))

### 7.2 Seleccion de la fuerza del prior (n0)

In [ ]:
def posterior(p, e=0.0, f=0.0, n0=4):
    # prior Beta(n0*p, n0*(1-p)) actualizado con e exitos y f fracasos locales
    a, b = n0 * p + e, n0 * (1 - p) + f
    return a / (a + b)

# evaluacion temporal: filas de 2016-17 con resultado del mismo distrito-cultivo en 2015-16
m["p_prior"] = oof_mix
ev = (m[m.campana_ini == 2015].set_index(["cod_ubigeo", "producto"]).exito.rename("e_prev"))
t = m[m.campana_ini == 2016].join(ev, on=["cod_ubigeo", "producto"]).dropna(subset=["e_prev"])
yt = t.exito.astype(int).values

res = {"sin evidencia": metricas(yt, t.p_prior.values)}
for n0 in [1, 2, 4, 8, 16]:
    pp = posterior(t.p_prior.values, t.e_prev.values, 1 - t.e_prev.values, n0)
    res[f"n0={n0}"] = metricas(yt, pp)
tabb = pd.DataFrame(res).T.round(3)
N0 = int(tabb.drop("sin evidencia").brier.idxmin().split("=")[1])
print("n0 elegido (menor Brier):", N0, "| filas evaluadas:", len(t))
tabb

### 7.3 Resumen de desempeno

In [ ]:
pp = posterior(t.p_prior.values, t.e_prev.values, 1 - t.e_prev.values, N0)
resumen = pd.DataFrame({
    "ML (todas las filas)":                metricas(y, oof_ml_sel),
    "Difuso (todas las filas)":           metricas(y, oof_fz),
    "Mezcla ML + difuso (todas)":         metricas(y, oof_mix),
    "Mezcla (filas con historial)":       metricas(yt, t.p_prior.values),
    "Mezcla + bayes (filas con historial)": metricas(yt, pp),
}).T.round(3)
resumen

### 7.4 Nivel de confianza (precision historica por nivel de probabilidad)

In [ ]:
from sklearn.isotonic import IsotonicRegression

def ajustar_conf(p, y_true):
    # precision observada (validacion) segun la distancia de la probabilidad a 0.5
    acierto = ((p >= .5).astype(int) == y_true).astype(float)
    return IsotonicRegression(increasing=True, out_of_bounds="clip").fit(np.abs(p - .5), acierto)

en_t = m.index.isin(t.index)
CONF_CON = ajustar_conf(pp, yt)                          # con historial local
CONF_SIN = ajustar_conf(oof_mix[~en_t], y[~en_t])        # sin historial local

xs = np.linspace(0, .5, 6)
pd.DataFrame({"|p - 0.5|": xs, "confianza sin historial": CONF_SIN.predict(xs),
              "confianza con historial": CONF_CON.predict(xs)}).round(2)

## 8. Ajuste final con todos los datos

In [ ]:
zona_f = Zona(m)
Xz = zona_f.transform(m)
wm_f = DifusoWM().fit(Xz, yrel)
cal_f = LogisticRegression().fit(wm_f.predict(Xz).reshape(-1, 1), y)
inner = list(GroupKFold(N_FOLDS).split(m, y, g))
ml_f = make_pipe(CAT, NUM, RF(**BEST))
if CALIBRAR:
    ml_f = CalibratedClassifierCV(ml_f, method="sigmoid", cv=inner)
ml_f.fit(m[FEATS], y)

print("Reglas difusas:", len(wm_f.reglas))
pd.DataFrame({"regla": [wm_f.texto(a) for a in wm_f.reglas]}).head(8)

## 9. Recomendador

In [ ]:
# Instantanea de la ultima campana completa (base para la siguiente campana)
BASE_INI = int(camp[camp.campana_completa].campana_ini.max())
cb = camp[camp.tipo_cultivo.eq("transitorio") & (camp.campana_ini == BASE_INI)]
TOT_D = cb.groupby("cod_ubigeo").siembra_ha.sum()
FAM_D = cb.groupby(["cod_ubigeo", "familia"]).siembra_ha.sum()
LEG_D = cb[cb.es_leguminosa].groupby("cod_ubigeo").siembra_ha.sum()
CUL_D = cb.groupby(["cod_ubigeo", "producto"]).siembra_ha.sum()
EV_D  = m[m.campana_ini == BASE_INI].set_index(["cod_ubigeo", "producto"]).exito

DIM = camp[["cod_ubigeo", "provincia", "distrito"]].drop_duplicates()
REF = (m.groupby(["provincia", "producto"])
       .agg(n=("exito", "size"), sup=("siembra_ha", "median"), share=("share_cultivo_dist", "median"),
            ingreso=("ingreso_ha_soles", "median"), familia=("familia", "first")).reset_index())

def recomendar(distrito, provincia=None, mes_siembra=None, familia_antecesor=None, top=10):
    q = DIM[DIM.distrito == distrito.upper()]
    if provincia: q = q[q.provincia == provincia.upper()]
    if len(q) != 1:
        raise ValueError(f"Distrito ambiguo o inexistente: {q.values.tolist()}")
    cod, prov = q.cod_ubigeo.iloc[0], q.provincia.iloc[0]

    c = REF[(REF.provincia == prov) & (REF.n >= MIN_PROV_REG)].copy()
    c["mes"] = mes_siembra if mes_siembra else c.producto.map(MES_TIPICO)
    c["mes_pico_siembra"] = c.mes
    c["mes_sin"], c["mes_cos"] = np.sin(2 * np.pi * c.mes / 12), np.cos(2 * np.pi * c.mes / 12)
    c["sin_siembra"] = 0
    c["dist_mes_tipico"] = dist_circular(c.mes, c.producto.map(MES_TIPICO))
    c["log_siembra"], c["share_cultivo_dist"] = np.log1p(c["sup"]), c["share"]
    tot = TOT_D.get(cod, 0)
    c["prev_share_familia"] = [FAM_D.get((cod, f), 0) / tot if tot > 0 else 0 for f in c.familia]
    c["prev_share_leguminosa"] = LEG_D.get(cod, 0) / tot if tot > 0 else 0
    c["log_prev_cultivo"] = [np.log1p(CUL_D.get((cod, p), 0)) for p in c.producto]

    p_ml = ml_f.predict_proba(c[FEATS])[:, 1]
    Xc = zona_f.transform(c)
    p_fz = cal_f.predict_proba(wm_f.predict(Xc).reshape(-1, 1))[:, 1]
    p0 = W_ML * p_ml + (1 - W_ML) * p_fz

    e_prev = np.array([EV_D.get((cod, p), np.nan) for p in c.producto])
    e = np.where(e_prev == 1, 1.0, 0.0); f = np.where(e_prev == 0, 1.0, 0.0)
    pp = posterior(p0, e, f, N0)
    hay = ~np.isnan(e_prev)
    conf = np.where(hay, CONF_CON.predict(np.abs(pp - .5)), CONF_SIN.predict(np.abs(pp - .5)))

    out = pd.DataFrame({
        "cultivo": c.producto.values, "familia": c.familia.values,
        "p_ml": p_ml, "p_difuso": p_fz, "prob_exito": pp, "confianza": conf,
        "historial_local": np.where(np.isnan(e_prev), "sin dato", np.where(e_prev == 1, "exito", "bajo")),
        "ingreso_ref_ha": c.ingreso.values,
        "reglas": [wm_f.top_reglas(Xc[i:i + 1], 1) for i in range(len(c))],
    })
    if familia_antecesor:
        out["alerta_rotacion"] = np.where(out.familia == familia_antecesor, "misma familia que el antecesor", "")
    out = out.sort_values("prob_exito", ascending=False).head(top).reset_index(drop=True)
    out.index += 1
    return out

cols = ["cultivo", "p_ml", "p_difuso", "prob_exito", "confianza", "historial_local", "ingreso_ref_ha"]
for d in ["TIABAYA", "MAJES"]:
    print("\n", d)
    display(recomendar(d)[cols].round(2).head(6))

In [ ]:
r = recomendar("TIABAYA", top=8).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(r.cultivo, r.prob_exito * 100, color="#4c9a2a", label="probabilidad de exito")
ax.scatter(r.confianza * 100, r.cultivo, color="k", zorder=3, label="confianza (precision historica)")
ax.set_xlim(0, 100); ax.set_xlabel("%"); ax.set_title("Recomendacion - Tiabaya"); ax.legend(loc="lower right", fontsize=8)
plt.show()
print(recomendar("TIABAYA", top=1).reglas.iloc[0])

## 10. Exportacion de artefactos

In [ ]:
ARTEFACTOS = dict(ml=ml_f, zona=zona_f, wm=wm_f, cal_wm=cal_f, conf_con=CONF_CON, conf_sin=CONF_SIN,
                  mes_tipico=MES_TIPICO.to_dict(), ref=REF, feats=FEATS, w_ml=W_ML, n0=N0, base_ini=BASE_INI)
joblib.dump(ARTEFACTOS, "agropredix_modelo.joblib")

with open("reglas_difusas.json", "w", encoding="utf-8") as f:
    json.dump([dict(regla=wm_f.texto(a)) for a in wm_f.reglas], f, ensure_ascii=False, indent=1)
resumen.to_csv("metricas_modelo.csv", encoding="utf-8-sig")
print(sorted(os.listdir(".")))

try:
    from google.colab import files
    for nombre in ["agropredix_modelo.joblib", "reglas_difusas.json", "metricas_modelo.csv"]:
        files.download(nombre)
except ImportError:
    pass